In [1]:
import torch
import torchvision

print("PyTorch 版本：", torch.__version__)
print("torchvision 版本：", torchvision.__version__)
print("GPU 可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 型号：", torch.cuda.get_device_name(0))

PyTorch 版本： 2.7.1+cu118
torchvision 版本： 0.22.1+cu118
GPU 可用： True
GPU 型号： NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import torch
import torch.nn as nn


class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # 残差分支：第一个卷积负责按需缩小宽高
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(num_features=out_channels)

        # 第二个卷积保持形状
        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(num_features=out_channels)

        self.relu = nn.ReLU()

        # 捷径：默认原样传递
        self.shortcut = nn.Identity()

        # 形状改变时，让捷径与残差分支匹配
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=1,
                    stride=stride,
                    padding=0,
                    bias=False,
                ),
                nn.BatchNorm2d(num_features=out_channels),
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

In [3]:
x = torch.randn(2, 64, 32, 32)

# 通道、宽高保持不变
block1 = BasicBlock(in_channels=64, out_channels=64, stride=1)
print("保持形状：", block1(x).shape)

# 通道增加，宽高减半
block2 = BasicBlock(in_channels=64, out_channels=128, stride=2)
print("改变形状：", block2(x).shape)

保持形状： torch.Size([2, 64, 32, 32])
改变形状： torch.Size([2, 128, 16, 16])


In [4]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # 开头：适配 CIFAR-10 的 32×32 图片
        self.stem = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
        )

        # 四个阶段，每个阶段两个残差块
        self.stage1 = nn.Sequential(
            BasicBlock(in_channels=64, out_channels=64, stride=1),
            BasicBlock(in_channels=64, out_channels=64, stride=1),
        )

        self.stage2 = nn.Sequential(
            BasicBlock(in_channels=64, out_channels=128, stride=2),
            BasicBlock(in_channels=128, out_channels=128, stride=1),
        )

        self.stage3 = nn.Sequential(
            BasicBlock(in_channels=128, out_channels=256, stride=2),
            BasicBlock(in_channels=256, out_channels=256, stride=1),
        )

        self.stage4 = nn.Sequential(
            BasicBlock(in_channels=256, out_channels=512, stride=2),
            BasicBlock(in_channels=512, out_channels=512, stride=1),
        )

        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
        self.fc = nn.Linear(
            in_features=512,
            out_features=num_classes,
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)

        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet18(num_classes=10).to(device)
x = torch.randn(2, 3, 32, 32, device=device)

model.eval()
with torch.no_grad():
    y = model(x)

print("运行设备：", device)
print("输出形状：", y.shape)

运行设备： cuda
输出形状： torch.Size([2, 10])


### 接下来加载真实图片。先完成下载数据、分批读取、检查形状，然后再写训练步骤。

In [8]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 把图片转成张量，并对各通道做归一化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5),
    ),
])

# 训练集：用于更新模型权重
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform,
)

# 测试集：用于之后评估模型
test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform,
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

print("训练图片数量：", len(train_dataset))
print("测试图片数量：", len(test_dataset))
print("类别名称：", train_dataset.classes)

训练图片数量： 50000
测试图片数量： 10000
类别名称： ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [9]:
# 衡量分类分数与正确标签之间的差距
criterion = nn.CrossEntropyLoss()

# 根据梯度更新模型参数
optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=0.0005,
)

In [10]:
# 切换到训练模式，BN 会使用当前批次的统计量
model.train()

# 取出一个批次，并放到与模型相同的设备
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

# 1. 清空之前的梯度
optimizer.zero_grad()

# 2. 前向传播，得到每张图片的 10 个类别分数
outputs = model(images)

# 3. 计算损失
loss = criterion(outputs, labels)

# 4. 反向传播，计算各参数的梯度
loss.backward()

# 5. 根据梯度更新参数
optimizer.step()

print("输出形状：", outputs.shape)
print("本批次损失：", loss.item())

predictions = outputs.argmax(dim=1)
accuracy = (predictions == labels).float().mean().item()
print(f"本批次准确率：{accuracy:.2%}")

输出形状： torch.Size([64, 10])
本批次损失： 2.462764263153076
本批次准确率：12.50%


In [11]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_index, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # 累计整轮的损失和预测正确数量
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

        # 定期打印，方便观察训练进度
        if (batch_index + 1) % 100 == 0:
            print(
                f"批次 {batch_index + 1}/{len(train_loader)} | "
                f"平均损失：{total_loss / total_samples:.4f} | "
                f"准确率：{total_correct / total_samples:.2%}"
            )

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )

In [12]:
train_loss, train_accuracy = train_one_epoch(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
)

print(
    f"本轮训练完成 | "
    f"损失：{train_loss:.4f} | "
    f"准确率：{train_accuracy:.2%}"
)

批次 100/782 | 平均损失：1.8333 | 准确率：33.69%
批次 200/782 | 平均损失：1.6965 | 准确率：38.36%
批次 300/782 | 平均损失：1.5945 | 准确率：42.36%
批次 400/782 | 平均损失：1.5049 | 准确率：45.51%
批次 500/782 | 平均损失：1.4348 | 准确率：48.20%
批次 600/782 | 平均损失：1.3740 | 准确率：50.47%
批次 700/782 | 平均损失：1.3162 | 准确率：52.70%
本轮训练完成 | 损失：1.2774 | 准确率：54.21%


In [13]:
def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )

In [14]:
test_loss, test_accuracy = evaluate(
    model=model,
    data_loader=test_loader,
    criterion=criterion,
    device=device,
)

print(
    f"测试损失：{test_loss:.4f} | "
    f"测试准确率：{test_accuracy:.2%}"
)

测试损失：0.8864 | 测试准确率：68.10%


In [15]:
from pathlib import Path

save_dir = Path("./checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "completed_epochs": 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
    save_dir / "resnet18_cifar10_epoch1.pth",
)

print("已保存第一轮训练进度")

已保存第一轮训练进度
